# Fase 16 — Adaptador híbrido 266 ↔ 513 (versão corrigida)

**Como executar:** selecione `Ambiente de execução → Executar tudo`.

O notebook usa uma única célula principal autocontida para impedir erros por execução fora de ordem. Ele não inicia o treinamento: valida o bridge RtF e somente libera o smoke test quando exibir `ADAPTER_APPROVED=true`.

> Use ambiente CPU com memória RAM padrão ou alta. A primeira compilação do Go pode demorar.

In [ ]:
"""
FASE 16 — Validação formal do adaptador ML (266) <-> RtF (513)

Esta fase NÃO executa treinamento. Ela valida a fronteira criptográfica e a
recuperação após falha antes de liberar o runner híbrido oficial.
"""

from __future__ import annotations

import hashlib
import json
import os
import platform
import random
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path


# -----------------------------------------------------------------------------
# 1. CONFIGURAÇÃO CONGELADA
# -----------------------------------------------------------------------------
REPO_URL = "https://github.com/KAIST-CryptLab/RtF-Transciphering.git"
REPO_COMMIT = "105fc73115b56f1d6ff357029c7682b19a6d8510"
ML_VECTOR_LENGTH = 266
RTF_VECTOR_LENGTH = 513
RUBATO_VARIANT = "RUBATO80S"
SEED = 42
TEST_TIMEOUT_SECONDS = 1800
MAX_ABS_ERROR_TOLERANCE = 1e-3
MEAN_ABS_ERROR_TOLERANCE = 1e-4
PADDING_ABS_ERROR_TOLERANCE = 1e-3
NUM_RANDOM_TESTS = 5

ROOT = Path("/content/phase16_adapter_validation_v2")
REPO = ROOT / "RtF-Transciphering"
RESULTS_LOCAL = ROOT / "results"
GO_TEST_FILE = REPO / "ckks_fv" / "phase16_adapter_test.go"

DRIVE_BASE = Path(
    "/content/drive/MyDrive/Mestrado_Criptografia/OFFICIAL_CAMPAIGN_V1/"
    "runs/PHYSIONET_CHALLENGE_2012_FINAL_V1/hybrid/"
    "v16_adapter_266_513_validation_v2"
)


def run(cmd, *, cwd=None, env=None, timeout=None, check=True):
    """Executa comando sem shell e devolve o resultado completo."""
    print("$", " ".join(map(str, cmd)))
    cp = subprocess.run(
        list(map(str, cmd)),
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
        check=False,
    )
    print(cp.stdout[-6000:])
    if check and cp.returncode != 0:
        raise RuntimeError(
            f"Comando falhou (exit={cp.returncode}): {' '.join(map(str, cmd))}"
        )
    return cp


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


print("=" * 100)
print("FASE 16 — ADAPTADOR HÍBRIDO 266 <-> 513")
print("=" * 100)


# -----------------------------------------------------------------------------
# 2. DRIVE, AMBIENTE E REPOSITÓRIO EXATO
# -----------------------------------------------------------------------------
try:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("Fora do Colab: resultados permanecerão no diretório local.")

ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_LOCAL.mkdir(parents=True, exist_ok=True)
if Path("/content/drive/MyDrive").exists():
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)

if shutil.which("go") is None or shutil.which("git") is None:
    run(["apt-get", "update"])
    run(["apt-get", "install", "-y", "golang-go", "git"])

if not REPO.exists():
    run(["git", "clone", REPO_URL, str(REPO)])
run(["git", "fetch", "--all", "--tags"], cwd=REPO)
run(["git", "checkout", "--force", REPO_COMMIT], cwd=REPO)
actual_commit = run(["git", "rev-parse", "HEAD"], cwd=REPO).stdout.strip()
if actual_commit != REPO_COMMIT:
    raise RuntimeError(f"Commit incorreto: {actual_commit}")


# -----------------------------------------------------------------------------
# 3. TESTE GO: UMA REQUISIÇÃO POR PROCESSO (ESTADO CRIPTOGRÁFICO NOVO)
# -----------------------------------------------------------------------------
GO_SOURCE = r'''package ckks_fv

import (
    "crypto/rand"
    "encoding/json"
    "fmt"
    "math"
    "os"
    "testing"
    "time"

    "github.com/ldsec/lattigo/v2/utils"
)

type phase16Request struct {
    TestID string    `json:"test_id"`
    Delta  []float64 `json:"delta"`
}

type phase16Response struct {
    TestID          string    `json:"test_id"`
    Recovered       []float64 `json:"recovered"`
    MaxAbsError     float64   `json:"max_abs_error"`
    MeanAbsError    float64   `json:"mean_abs_error"`
    MaxPaddingError float64   `json:"max_padding_error"`
    WallSeconds     float64   `json:"wall_seconds"`
    VectorLength    int       `json:"vector_length"`
    RubatoVariant   string    `json:"rubato_variant"`
    BridgeVersion   string    `json:"bridge_version"`
}

func TestPhase16AdapterOnce(t *testing.T) {
    raw := os.Getenv("PHASE16_REQUEST_JSON")
    if raw == "" {
        t.Fatal("PHASE16_REQUEST_JSON ausente")
    }
    var req phase16Request
    if err := json.Unmarshal([]byte(raw), &req); err != nil {
        t.Fatalf("JSON inválido: %v", err)
    }
    if len(req.Delta) != 513 {
        t.Fatalf("expected 513 values, got %d", len(req.Delta))
    }

    start := time.Now()
    rubatoParam := RUBATO80S
    blocksize := RubatoParams[rubatoParam].Blocksize
    numRound := RubatoParams[rubatoParam].NumRound
    plainModulus := RubatoParams[rubatoParam].PlainModulus
    sigma := RubatoParams[rubatoParam].Sigma

    hbtpParams := RtFRubatoParams[0]
    params, err := hbtpParams.Params()
    if err != nil {
        t.Fatal(err)
    }
    params.SetPlainModulus(plainModulus)
    params.SetLogFVSlots(params.LogN())
    messageScaling := float64(params.PlainModulus()) / hbtpParams.MessageRatio

    rubatoModDown := RubatoModDownParams[rubatoParam].CipherModDown
    stcModDown := RubatoModDownParams[rubatoParam].StCModDown

    kgen := NewKeyGenerator(params)
    sk, pk := kgen.GenKeyPairSparse(hbtpParams.H)
    fvEncoder := NewMFVEncoder(params)
    ckksEncoder := NewCKKSEncoder(params)
    fvEncryptor := NewMFVEncryptorFromPk(params, pk)
    ckksDecryptor := NewCKKSDecryptor(params, sk)

    rotationsHalfBoot := kgen.GenRotationIndexesForHalfBoot(params.LogSlots(), hbtpParams)
    pDcds := fvEncoder.GenSlotToCoeffMatFV(2)
    rotationsStC := kgen.GenRotationIndexesForSlotsToCoeffsMat(pDcds)
    rotations := append(rotationsHalfBoot, rotationsStC...)
    rotkeys := kgen.GenRotationKeysForRotations(rotations, true, sk)
    rlk := kgen.GenRelinearizationKey(sk)
    hbtpKey := BootstrappingKey{Rlk: rlk, Rtks: rotkeys}

    hbtp, err := NewHalfBootstrapper(params, hbtpParams, hbtpKey)
    if err != nil {
        t.Fatal(err)
    }
    fvEvaluator := NewMFVEvaluator(
        params,
        EvaluationKey{Rlk: rlk, Rtks: rotkeys},
        pDcds,
    )

    key := make([]uint64, blocksize)
    for i := range key {
        key[i] = uint64(i + 1)
    }
    rubato := NewMFVRubato(
        rubatoParam, params, fvEncoder, fvEncryptor, fvEvaluator, rubatoModDown[0],
    )
    kCt := rubato.EncKey(key)

    data := make([]float64, params.N())
    copy(data[:513], req.Delta)

    nonces := make([][]byte, params.N())
    for i := 0; i < params.N(); i++ {
        nonces[i] = make([]byte, 8)
        if _, err := rand.Read(nonces[i]); err != nil {
            t.Fatal(err)
        }
    }
    counter := make([]byte, 8)
    if _, err := rand.Read(counter); err != nil {
        t.Fatal(err)
    }

    keystream := make([][]uint64, params.N())
    for i := 0; i < params.N(); i++ {
        keystream[i] = plainRubato(
            blocksize, numRound, nonces[i], counter, key, plainModulus, sigma,
        )
    }

    coeffs := make([]float64, params.N())
    for i := 0; i < params.N()/2; i++ {
        j := utils.BitReverse64(uint64(i), uint64(params.LogN()-1))
        coeffs[j] = data[i]
        coeffs[j+uint64(params.N()/2)] = data[i+params.N()/2]
    }

    plainRingT := ckksEncoder.EncodeCoeffsRingTNew(coeffs, messageScaling)
    poly := plainRingT.Value()[0]
    for i := 0; i < params.N(); i++ {
        j := utils.BitReverse64(uint64(i), uint64(params.LogN()))
        poly.Coeffs[0][j] = (poly.Coeffs[0][j] + keystream[i][0]) % params.PlainModulus()
    }

    plaintext := NewPlaintextFVLvl(params, 0)
    fvEncoder.FVScaleUp(plainRingT, plaintext)
    fvKeystreams := rubato.Crypt(nonces, counter, kCt, rubatoModDown)
    fvKS := fvEvaluator.SlotsToCoeffs(fvKeystreams[0], stcModDown)

    // Guarda explícita para impedir o panic que invalidava o bridge persistente.
    level := fvKS.Level()
    if level <= 0 {
        t.Fatalf("INVALID_FVKS_LEVEL:%d", level)
    }
    fvEvaluator.ModSwitchMany(fvKS, fvKS, level)

    ciphertext := NewCiphertextFVLvl(params, 1, 0)
    ciphertext.Value()[0] = plaintext.Value()[0].CopyNew()
    fvEvaluator.Sub(ciphertext, fvKS, ciphertext)
    fvEvaluator.TransformToNTT(ciphertext, ciphertext)
    ciphertext.SetScale(math.Exp2(math.Round(math.Log2(
        float64(params.Qi()[0]) / float64(params.PlainModulus()) * messageScaling,
    ))))

    ctBoot, _ := hbtp.HalfBoot(ciphertext, false)
    values := ckksEncoder.DecodeComplex(
        ckksDecryptor.DecryptNew(ctBoot), params.LogSlots(),
    )

    recovered := make([]float64, 513)
    maxErr := 0.0
    meanErr := 0.0
    maxPaddingErr := 0.0
    for i := 0; i < 513; i++ {
        recovered[i] = real(values[i])
        e := math.Abs(recovered[i] - req.Delta[i])
        meanErr += e
        if e > maxErr {
            maxErr = e
        }
        if i >= 266 && math.Abs(recovered[i]) > maxPaddingErr {
            maxPaddingErr = math.Abs(recovered[i])
        }
    }
    meanErr /= 513.0

    resp := phase16Response{
        TestID: req.TestID,
        Recovered: recovered,
        MaxAbsError: maxErr,
        MeanAbsError: meanErr,
        MaxPaddingError: maxPaddingErr,
        WallSeconds: time.Since(start).Seconds(),
        VectorLength: 513,
        RubatoVariant: "RUBATO80S",
        BridgeVersion: "phase16_fresh_process_v2",
    }
    b, err := json.Marshal(resp)
    if err != nil {
        t.Fatal(err)
    }
    fmt.Printf("PHASE16_JSON:%s\n", string(b))
}
'''

GO_TEST_FILE.write_text(GO_SOURCE, encoding="utf-8")
run(["gofmt", "-w", str(GO_TEST_FILE)])
run(["go", "test", "./ckks_fv", "-run", "^$"], cwd=REPO, timeout=600)


# -----------------------------------------------------------------------------
# 4. ADAPTADOR EXPLÍCITO E VETORES SENTINELA
# -----------------------------------------------------------------------------
def pack_ml_to_rtf(delta_ml):
    if len(delta_ml) != ML_VECTOR_LENGTH:
        raise ValueError(f"Esperados {ML_VECTOR_LENGTH} parâmetros de ML")
    values = [float(x) for x in delta_ml]
    if not all(x == x and abs(x) != float("inf") for x in values):
        raise ValueError("Update contém NaN ou infinito")
    return values + [0.0] * (RTF_VECTOR_LENGTH - ML_VECTOR_LENGTH)


def unpack_rtf_to_ml(recovered):
    if len(recovered) != RTF_VECTOR_LENGTH:
        raise ValueError(f"Resposta RtF inválida: {len(recovered)} posições")
    return [float(x) for x in recovered[:ML_VECTOR_LENGTH]]


def build_vectors():
    rng = random.Random(SEED)
    vectors = []
    vectors.append(("zeros", [0.0] * ML_VECTOR_LENGTH))
    vectors.append(("ramp", [(i - 133) / 10000.0 for i in range(ML_VECTOR_LENGTH)]))
    impulse_a = [0.0] * ML_VECTOR_LENGTH
    impulse_a[0] = 0.03125
    vectors.append(("impulse_first", impulse_a))
    impulse_b = [0.0] * ML_VECTOR_LENGTH
    impulse_b[-1] = -0.03125
    vectors.append(("impulse_last", impulse_b))
    alternating = [((-1.0) ** i) * (i + 1) / 100000.0 for i in range(ML_VECTOR_LENGTH)]
    vectors.append(("alternating_unique", alternating))
    for idx in range(NUM_RANDOM_TESTS):
        vectors.append((f"random_{idx:02d}", [rng.uniform(-0.05, 0.05) for _ in range(ML_VECTOR_LENGTH)]))
    return vectors


def execute_one(test_id, delta_513, *, expect_success=True):
    request = {"test_id": test_id, "delta": delta_513}
    env = os.environ.copy()
    env["PHASE16_REQUEST_JSON"] = json.dumps(request, separators=(",", ":"))
    started = time.time()
    cp = run(
        ["go", "test", "./ckks_fv", "-run", "^TestPhase16AdapterOnce$", "-count=1", "-v", "-timeout=0"],
        cwd=REPO,
        env=env,
        timeout=TEST_TIMEOUT_SECONDS,
        check=False,
    )
    if expect_success and cp.returncode != 0:
        raise RuntimeError(f"Teste criptográfico {test_id} falhou")
    if not expect_success:
        return {"test_id": test_id, "failed_as_expected": cp.returncode != 0, "exit_code": cp.returncode}
    marker = "PHASE16_JSON:"
    lines = [line for line in cp.stdout.splitlines() if marker in line]
    if len(lines) != 1:
        raise RuntimeError(f"Resposta JSON ausente ou duplicada em {test_id}")
    response = json.loads(lines[0].split(marker, 1)[1])
    response["process_wall_seconds"] = time.time() - started
    return response


# -----------------------------------------------------------------------------
# 5. TESTE DE FALHA + RECUPERAÇÃO EM PROCESSO NOVO
# -----------------------------------------------------------------------------
failure_probe = execute_one(
    "intentional_bad_length",
    [0.0] * ML_VECTOR_LENGTH,
    expect_success=False,
)
if not failure_probe["failed_as_expected"]:
    raise RuntimeError("O bridge aceitou indevidamente um vetor com tamanho inválido")

recovery_vector = pack_ml_to_rtf([0.001 * ((i % 7) - 3) for i in range(ML_VECTOR_LENGTH)])
recovery_response = execute_one("recovery_after_failure", recovery_vector)
recovery_after_failure_ok = len(recovery_response["recovered"]) == RTF_VECTOR_LENGTH


# -----------------------------------------------------------------------------
# 6. CAMPANHA DE VALIDAÇÃO DO ADAPTADOR
# -----------------------------------------------------------------------------
test_results = []
for test_id, delta_ml in build_vectors():
    packed = pack_ml_to_rtf(delta_ml)
    response = execute_one(test_id, packed)
    recovered_ml = unpack_rtf_to_ml(response["recovered"])

    ml_errors = [abs(a - b) for a, b in zip(delta_ml, recovered_ml)]
    max_ml_error = max(ml_errors)
    mean_ml_error = sum(ml_errors) / len(ml_errors)
    padding_errors = [abs(x) for x in response["recovered"][ML_VECTOR_LENGTH:]]
    max_padding_error = max(padding_errors)

    passed = (
        max_ml_error <= MAX_ABS_ERROR_TOLERANCE
        and mean_ml_error <= MEAN_ABS_ERROR_TOLERANCE
        and max_padding_error <= PADDING_ABS_ERROR_TOLERANCE
        and response["vector_length"] == RTF_VECTOR_LENGTH
        and response["rubato_variant"] == RUBATO_VARIANT
    )
    test_results.append(
        {
            "test_id": test_id,
            "passed": passed,
            "max_ml_error": max_ml_error,
            "mean_ml_error": mean_ml_error,
            "max_padding_error": max_padding_error,
            "crypto_max_abs_error_513": response["max_abs_error"],
            "crypto_mean_abs_error_513": response["mean_abs_error"],
            "crypto_wall_seconds": response["wall_seconds"],
            "process_wall_seconds": response["process_wall_seconds"],
        }
    )


# -----------------------------------------------------------------------------
# 7. VEREDITO, AUDITORIA E PERSISTÊNCIA
# -----------------------------------------------------------------------------
adapter_approved = (
    failure_probe["failed_as_expected"]
    and recovery_after_failure_ok
    and len(test_results) >= 10
    and all(item["passed"] for item in test_results)
)

report = {
    "schema_version": "phase16_adapter_validation_v2",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "adapter_approved": adapter_approved,
    "training_executed": False,
    "dataset_files_consumed": False,
    "next_authorized_step": (
        "PHYSIONET_HYBRID_SMOKE_TEST_1_ROUND_5_CLIENTS"
        if adapter_approved
        else "CORRECT_ADAPTER_OR_CRYPTO_BOUNDARY_AND_REPEAT_PHASE16"
    ),
    "configuration": {
        "repository": REPO_URL,
        "commit": actual_commit,
        "ml_vector_length": ML_VECTOR_LENGTH,
        "rtf_vector_length": RTF_VECTOR_LENGTH,
        "rubato_variant": RUBATO_VARIANT,
        "seed": SEED,
        "max_abs_error_tolerance": MAX_ABS_ERROR_TOLERANCE,
        "mean_abs_error_tolerance": MEAN_ABS_ERROR_TOLERANCE,
        "padding_abs_error_tolerance": PADDING_ABS_ERROR_TOLERANCE,
    },
    "environment": {
        "python": sys.version,
        "platform": platform.platform(),
        "go_version": run(["go", "version"]).stdout.strip(),
    },
    "artifacts": {
        "go_test_sha256": sha256_file(GO_TEST_FILE),
    },
    "failure_probe": failure_probe,
    "recovery_after_failure_ok": recovery_after_failure_ok,
    "recovery_response_metrics": {
        "max_abs_error": recovery_response["max_abs_error"],
        "mean_abs_error": recovery_response["mean_abs_error"],
        "max_padding_error": recovery_response["max_padding_error"],
    },
    "tests": test_results,
}

report_path = RESULTS_LOCAL / "phase16_adapter_validation_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

csv_path = RESULTS_LOCAL / "phase16_adapter_validation_metrics.csv"
columns = [
    "test_id", "passed", "max_ml_error", "mean_ml_error",
    "max_padding_error", "crypto_max_abs_error_513",
    "crypto_mean_abs_error_513", "crypto_wall_seconds", "process_wall_seconds",
]
with csv_path.open("w", encoding="utf-8", newline="") as f:
    f.write(",".join(columns) + "\n")
    for row in test_results:
        f.write(",".join(str(row[c]) for c in columns) + "\n")

if Path("/content/drive/MyDrive").exists():
    shutil.copy2(report_path, DRIVE_BASE / report_path.name)
    shutil.copy2(csv_path, DRIVE_BASE / csv_path.name)
    shutil.copy2(GO_TEST_FILE, DRIVE_BASE / GO_TEST_FILE.name)

print("\n" + "=" * 100)
print(json.dumps({
    "ADAPTER_APPROVED": adapter_approved,
    "tests_passed": sum(x["passed"] for x in test_results),
    "tests_total": len(test_results),
    "failure_rejected": failure_probe["failed_as_expected"],
    "recovery_after_failure": recovery_after_failure_ok,
    "next_authorized_step": report["next_authorized_step"],
    "report": str(report_path),
    "drive_copy": str(DRIVE_BASE) if Path("/content/drive/MyDrive").exists() else None,
}, indent=2))
print("=" * 100)

if not adapter_approved:
    raise RuntimeError(
        "ADAPTER_APPROVED=false. O treinamento híbrido continua bloqueado; "
        "consulte o relatório JSON para identificar o teste reprovado."
    )

print("ADAPTER_APPROVED=true — liberado apenas o smoke test híbrido de 1 rodada e 5 clientes.")
